In [ ]:
# [NB02-C01] SETUP
# What: install the pinned pm4py version and the powerlaw package, import
#       all the libraries used in this notebook, create the figures folder.
# Why: powerlaw is needed for the distribution analysis of the variants;
#      versions are pinned so Colab and the local environment agree.
# Expected: no errors (on Colab the installs take about one minute).

%pip install pm4py==2.7.23.1 powerlaw -q

import os                               # filesystem helpers
import pandas as pd                     # dataframes and CSV handling
import numpy as np                      # numeric helpers
import matplotlib.pyplot as plt         # plots
import seaborn as sns                   # nicer statistical plots
import pm4py                            # process mining library of the course
import powerlaw                         # power-law / Pareto distribution fit
from scipy.stats import kruskal, mannwhitneyu, spearmanr   # non-parametric tests (chosen in NB01)
from scipy.stats import poisson, lognorm                    # distribution models for the tail analysis

# Create the folder where the report figures will be saved (safe if it exists)
os.makedirs('figures', exist_ok=True)

In [ ]:
# [NB02-C02] LOAD THE PREPARED FILES FROM NB01
# What: load the three artifacts produced by NB01: the prepared raw log,
#       the abstract control-flow view and the case table.
# Why: NB01 already audited and standardised the data; reloading its
#      exports keeps this notebook runnable on its own without repeating
#      the cleaning (same pattern as the Case8 notebooks of the course).
#      NO row was dropped in NB01 and none will be dropped here.
# Expected: 25,115 raw rows, 16,826 abstract rows, 1,820 stays.

# Paths of the prepared files (on Colab: upload them and update the paths)
raw_path = 'prepared_event_log.csv'
abstract_path = 'abstract_event_log.csv'
cases_path = 'case_table.csv'

# Defensive check: stop with a clear message if a file is missing
for p in [raw_path, abstract_path, cases_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(p + ' not found - run NB01_Preprocessing first.')

# Load the prepared raw event log (dates parsed, case ids as strings)
raw_log = pd.read_csv(raw_path, parse_dates=['time:timestamp'])
raw_log['case:concept:name'] = raw_log['case:concept:name'].astype(str)

# Load the abstract control-flow view
abstract_log = pd.read_csv(abstract_path, parse_dates=['time:timestamp'])
abstract_log['case:concept:name'] = abstract_log['case:concept:name'].astype(str)

# Load the case table (one row per stay)
case_table = pd.read_csv(cases_path, index_col=0, parse_dates=['start_time', 'end_time'])
case_table.index = case_table.index.astype(str)

# Verify the sizes against what NB01 exported: nothing may be lost
print('Raw view rows: {} (expected 25,115)'.format(len(raw_log)))
print('Abstract view rows: {} (expected 16,826)'.format(len(abstract_log)))
print('Stays in the case table: {} (expected 1,820)'.format(len(case_table)))

In [ ]:
# [NB02-C03] SANITY CHECK: START AND END ACTIVITIES
# What: re-check start and end activities on both views after the reload.
# Why: standard course practice - after every load or transformation the
#      log boundaries are verified, never assumed.
# Expected: 1,820 "Enter the ED" starts and 1,820 "Discharge from the ED"
#      ends on both views.

# Start/end activities on the raw view
print('Raw view start activities: {}'.format(pm4py.get_start_activities(raw_log)))
print('Raw view end activities: {}'.format(pm4py.get_end_activities(raw_log)))

# Start/end activities on the abstract view
print('Abstract view start activities: {}'.format(pm4py.get_start_activities(abstract_log)))
print('Abstract view end activities: {}'.format(pm4py.get_end_activities(abstract_log)))

# Interpretation: both views keep the expected boundaries, so the reload
# preserved the log integrity and all 1,820 stays are still complete.

In [ ]:
# [NB02-C04] VOLUME AND DURATION RECAP
# What: recap the volume of the process and the stay duration statistics
#       computed in NB01, as the baseline of the performance analysis.
# Why: every performance comparison below refers to these global numbers;
#      after NB01 we know the distribution is right-skewed, so median and
#      percentiles (in minutes) are the reference statistics, not the mean.
# Expected: median 301 min, p90 748.5 min.

# Global volume of the process
print('Stays: {}'.format(len(case_table)))
print('Events (raw view): {}'.format(len(raw_log)))
print('Median events per stay: {:.0f}'.format(case_table['event_count'].median()))

# Stay duration statistics, in minutes and hours (units always explicit)
lead = case_table['lead_time_min']
print('\nStay duration (lead time):')
print('  median: {:.1f} min ({:.1f} h)'.format(lead.median(), lead.median() / 60))
print('  IQR: {:.1f} - {:.1f} min'.format(lead.quantile(0.25), lead.quantile(0.75)))
print('  p90: {:.1f} min ({:.1f} h)'.format(lead.quantile(0.90), lead.quantile(0.90) / 60))
print('  p95: {:.1f} min ({:.1f} h)'.format(lead.quantile(0.95), lead.quantile(0.95) / 60))
print('  max: {:.1f} min ({:.1f} h)'.format(lead.max(), lead.max() / 60))

# Robust dispersion (course note): on a skewed, non-normal distribution the
# standard deviation is unreliable, so we add the median absolute deviation
# (MAD) next to the IQR. Both resist the long tail.
mad = (lead - lead.median()).abs().median()
print('\nMedian absolute deviation (MAD): {:.1f} min'.format(mad))

# Normal operating range (course note): define the "normal" band with the
# Tukey fences [Q1 - 1.5*IQR, Q3 + 1.5*IQR]. Stays outside it are FLAGGED,
# never removed - the no-data-loss rule: an outlier duration is a signal, not
# noise, and stays in the data and in every later analysis.
q1 = lead.quantile(0.25)
q3 = lead.quantile(0.75)
iqr = q3 - q1
lower_fence = max(0, q1 - 1.5 * iqr)
upper_fence = q3 + 1.5 * iqr
outside = (lead < lower_fence) | (lead > upper_fence)
print('Normal operating range (Tukey fences): {:.0f} - {:.0f} min'.format(lower_fence, upper_fence))
print('Stays outside the normal range (flagged, kept): {} ({:.1f}%)'.format(
    int(outside.sum()), outside.mean() * 100))
case_table['duration_outlier'] = outside

# Interpretation: half of the patients complete the stay within ~5 hours,
# but one stay in ten lasts more than ~12.5 hours: the tail is where the
# performance analysis must look. The stays beyond the upper Tukey fence are
# flagged as duration outliers and kept: they are the operational cases the
# analysis is meant to surface, not data to discard.

In [ ]:
# [NB02-C05] GOALS AND KPI DEFINITION (LECTURE 1.3 FRAMEWORK)
# What: restate the strategic / tactical / operational goals of the ED and
#       define the KPIs of this analysis with the course framework: each
#       KPI = item measured + unit of measure + population + timing, over
#       the three KPI categories (time, cost, delivered quality).
# Why: performance measurement starts from the goals, not from the data
#      (lecture 1.3): every number computed in this notebook must serve a
#      declared KPI, and every KPI must trace back to a goal.
# Expected: a compact KPI definition table, reused by the report.

# Goals of the organisation (defined in the report, restated here so that
# the notebook is self-contained):
## Strategic:   S1 efficient patient flow for better outcomes and safety;
##              S2 regulatory compliance and equity of treatment.
## Tactical:    T1 reduce stay duration (median/p90) without penalising
##              urgent patients; T2 flows appropriate to acuity and
##              disposition; T3 reduce structural deviations.
## Operational: O1 monitor elapsed times between activities; O2 monitor
##              clinical repetitions; O3 monitor per-group KPIs.

# KPI definitions with the framework: item + unit + population + timing
kpi_definitions = pd.DataFrame([
    {'kpi': 'Stay lead time', 'category': 'time', 'goal': 'T1',
     'item': 'duration from Enter the ED to Discharge from the ED',
     'unit': 'minutes', 'population': 'all 1,820 stays (and per segment)',
     'timing': 'measured at discharge; reported as median and p90'},
    {'kpi': 'Transition elapsed time', 'category': 'time', 'goal': 'O1',
     'item': 'time between consecutive events of a stay',
     'unit': 'minutes', 'population': '15,006 transitions (abstract view)',
     'timing': 'per transition; median and p90 per activity pair'},
    {'kpi': 'Incremental lead time', 'category': 'time', 'goal': 'T1 / O1',
     'item': 'cumulative time from arrival to each activity milestone',
     'unit': 'minutes', 'population': 'first occurrence per stay of each activity',
     'timing': 'per stay; median and p90 per milestone'},
    {'kpi': 'Clinical workload (cost proxy)', 'category': 'cost', 'goal': 'O2',
     'item': 'recordings of reconciliation / dispensations / vital checks',
     'unit': 'count per stay', 'population': 'all stays (raw view)',
     'timing': 'counted at discharge; median per segment'},
    {'kpi': 'Interrupted pathway rate', 'category': 'quality', 'goal': 'S2 / T2',
     'item': 'stays ending in LWBS / eloped / AMA / expired / other',
     'unit': '% of stays', 'population': 'all 1,820 stays',
     'timing': 'at discharge; overall and per segment'},
    {'kpi': 'Urgent-path timeliness', 'category': 'quality', 'goal': 'T2',
     'item': 'lead time gap between acuity 1-3 and acuity 4-5 stays',
     'unit': 'minutes (median gap)', 'population': '1,764 stays with recorded acuity',
     'timing': 'at discharge; per acuity level'},
])

# Save the KPI table for the report
kpi_definitions.to_csv('kpi_definitions.csv', index=False)
print('Saved kpi_definitions.csv')

kpi_definitions

# Interpretation: no monetary cost is recorded in the log, so the cost
# category is covered by a declared proxy (clinical workload per stay).
# Every analysis below references one of these KPIs.

In [ ]:
# [NB02-C06] WHY PROCESSING TIME CANNOT BE COMPUTED - AND THE CHOSEN STRATEGY
# What: state explicitly why the service/processing time is not computable
#       on this log, list the three fallback strategies of the course
#       (lecture 1.3) and declare the one adopted.
# Why: each event carries a single timestamp (no lifecycle start/complete):
#      the time between two events mixes waiting and working, so the true
#      processing time of an activity cannot be isolated. The choice among
#      the fallback strategies changes what we can claim, so it was
#      discussed with and approved by the project owner.
# Expected: no computation here - this cell documents the decision.

# The three options from the course (lecture 1.3 - Performance Measurement):
## (a) processing time from the timestamp of the NEXT activity: works only
##     for very regular processes and does not reveal the waiting time;
##     unsafe here because clinical recordings run in parallel at the same
##     minute, so the "next" event often is not the completion of the
##     current activity.
## (b) lead time BY CASE (first to last event): does not give the cycle
##     time, but allows studying the relation between case lead time and
##     case COMPLEXITY.
## (c) INCREMENTAL lead time (cumulative time to each activity): does not
##     give the cycle time, but allows identifying the BOTTLENECKS.

# DECISION (approved by the project owner): strategy (c) is adopted as the
# primary tool for bottleneck identification [NB02-C09], complemented by
# strategy (b) to relate duration and case complexity [NB02-C10].
# Option (a) is NOT adopted: with parallel same-minute recordings it would
# mislabel waiting as processing. Consequently, the gap to the next event
# computed in [NB02-C07] is always called elapsed/waiting time and never
# processing time, in the code as in the report.
print('Adopted strategy: (c) incremental lead time + (b) lead time vs case complexity')

In [ ]:
# [NB02-C07] ELAPSED TIME BETWEEN CONSECUTIVE EVENTS
# What: compute the time elapsed between each event and the next one of
#       the same stay, on the abstract view.
# Why: this is the elapsed/waiting time defined in [NB02-C06] (never the
#      processing time). The abstract view is used because repeated
#      same-instant recordings would only add zero gaps.
# Expected: many short gaps and a long right tail, in minutes.

# Sort defensively (the view is already sorted) and compute the gap to the
# next event inside each stay
abstract_log = abstract_log.sort_values(['case:concept:name', 'time:timestamp', 'source_row'])
abstract_log['next_activity'] = abstract_log.groupby('case:concept:name')['concept:name'].shift(-1)
abstract_log['next_time'] = abstract_log.groupby('case:concept:name')['time:timestamp'].shift(-1)
abstract_log['elapsed_to_next_min'] = (abstract_log['next_time'] - abstract_log['time:timestamp']).dt.total_seconds() / 60

# Describe the elapsed times (last event of each stay has no successor)
gaps = abstract_log['elapsed_to_next_min'].dropna()
print('Transitions measured: {}'.format(len(gaps)))
print('Elapsed time to next event: median {:.1f} min, p90 {:.1f} min, max {:.1f} min'.format(
    gaps.median(), gaps.quantile(0.90), gaps.max()))

# Interpretation: elapsed time between events includes both waiting and
# working, because the log does not distinguish them; the bottleneck
# reading is done on the incremental lead time of [NB02-C09].

In [ ]:
# [NB02-C08] TRANSITION PERFORMANCE TABLE
# What: for every pair (activity -> next activity), count how often it
#       occurs and compute the median and 90th percentile elapsed time.
# Why: this table localises WHERE the time is spent: a transition with a
#      high median (or a huge p90) is a queue or a slow hand-over.
# Expected: the transition into "Discharge from the ED" and the loops on
#      clinical activities concentrate most of the elapsed time.

# Group by transition and aggregate count / median / p90 (minutes)
def p90(values):
    return values.quantile(0.90)
transitions = abstract_log.dropna(subset=['next_activity']).groupby(['concept:name', 'next_activity'])
transition_table = transitions['elapsed_to_next_min'].agg(['size', 'median', p90]).reset_index()
transition_table.columns = ['activity', 'next_activity', 'count', 'median_elapsed_min', 'p90_elapsed_min']

# Sort by frequency to read the main flow first
transition_table = transition_table.sort_values('count', ascending=False).reset_index(drop=True)

# Save the table for the report
transition_table.to_csv('transition_performance.csv', index=False)
print('Saved transition_performance.csv ({} transitions)'.format(len(transition_table)))

transition_table

# Interpretation: the table is read together with the performance DFG of
# [NB02-C12]; the transitions with high median elapsed time are candidate
# bottlenecks, confirmed on the milestone timeline of [NB02-C09].

In [ ]:
# [NB02-C09] INCREMENTAL LEAD TIME AND BOTTLENECKS (STRATEGY C)
# What: for each stay, compute the cumulative time from arrival to the
#       FIRST occurrence of each activity (the process milestones), then
#       look for the largest time jumps between consecutive milestones.
# Why: with single timestamps this is the course-approved way to localise
#      bottlenecks (lecture 1.3, option c): the milestone whose incremental
#      time grows the most is the step patients wait for.
# Expected: triage almost immediate after arrival; medication and
#      discharge milestones much later in the stay.

# Minutes from the start of the stay for every event
abstract_log['case_start'] = abstract_log.groupby('case:concept:name')['time:timestamp'].transform('min')
abstract_log['minutes_from_start'] = (abstract_log['time:timestamp'] - abstract_log['case_start']).dt.total_seconds() / 60

# First occurrence of each activity inside each stay
first_occurrence = abstract_log.groupby(['case:concept:name', 'concept:name'])['minutes_from_start'].min().reset_index()

# Median and p90 incremental lead time per activity milestone, with the
# number of stays in which the activity occurs (denominator)
milestones = first_occurrence.groupby('concept:name')['minutes_from_start'].agg(['size', 'median', p90]).reset_index()
milestones.columns = ['activity', 'stays', 'median_min_from_start', 'p90_min_from_start']
milestones = milestones.sort_values('median_min_from_start').reset_index(drop=True)

# Save the milestone table for the report
milestones.to_csv('incremental_lead_time.csv', index=False)
print('Saved incremental_lead_time.csv')

# Quality check of the strategy: milestone order must be consistent with
# the process (Enter at 0, Discharge at the lead time) - we verify it
enter_median = milestones[milestones['activity'] == 'Enter the ED']['median_min_from_start'].iloc[0]
discharge_median = milestones[milestones['activity'] == 'Discharge from the ED']['median_min_from_start'].iloc[0]
print('Check: Enter the ED median = {:.0f} min (expected 0); Discharge median = {:.0f} min (close to the 301 min median lead time)'.format(
    enter_median, discharge_median))

milestones

# Interpretation: reading the median column top-down gives the typical
# timeline of a stay. The largest jump between consecutive milestones is
# the main bottleneck candidate; it is challenged per patient group in the
# comparison cells below, because a bottleneck that only hits some segments
# is a different problem than a global one.

In [ ]:
# [NB02-C10] LEAD TIME VS CASE COMPLEXITY (STRATEGY B)
# What: relate the stay duration to the case complexity, measured by the
#       number of events and the number of diagnoses of the stay.
# Why: strategy (b) of lecture 1.3: without a cycle time we can still ask
#      whether long stays are long because MORE happens (complexity) or
#      because the same things happen SLOWER (waiting) - two different
#      improvement levers.
# Expected: a positive but imperfect correlation: part of the duration is
#      not explained by complexity.

# Spearman correlation (rank-based: durations are not normal, NB01)
rho_events, p_events = spearmanr(case_table['event_count'], case_table['lead_time_min'])
print('Lead time vs number of events: Spearman rho = {:.3f} (p = {:.4f})'.format(rho_events, p_events))
rho_diag, p_diag = spearmanr(case_table['diagnosis_count'], case_table['lead_time_min'])
print('Lead time vs number of diagnoses: Spearman rho = {:.3f} (p = {:.4f})'.format(rho_diag, p_diag))

# Scatter plot of complexity vs duration
plt.figure(figsize=(9, 5))
plt.scatter(case_table['event_count'], case_table['lead_time_min'] / 60, s=8, alpha=0.35, color='#1f4e79')
plt.xlabel('Events per stay (case complexity)')
plt.ylabel('Stay duration (hours)')
plt.title('Lead time vs case complexity')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB02_lead_vs_complexity.png', dpi=200)
plt.show()

# Interpretation: the correlation is positive and significant but far from
# perfect: stays with the same number of events show very different
# durations, so waiting - not only workload - drives part of the lead
# time. This is quantified variant by variant in [NB02-C18].

In [ ]:
# [NB02-C11] FREQUENCY DFG
# What: discover and visualise the Directly-Follows Graph with the
#       frequency of each edge, on the abstract view.
# Why: the DFG is the exploratory map of the process: it shows the real
#      flows and their volumes before any model is discovered (NB03).
# Expected: a main path Enter -> Triage -> clinical activities -> Discharge
#      with loops on the clinical activities.

# Discover the DFG (edges with frequencies, plus start/end activities)
dfg, start_activities, end_activities = pm4py.discover_dfg(abstract_log)

# Save the figure for the report, then view it inline
pm4py.save_vis_dfg(dfg, start_activities, end_activities, 'figures/NB02_frequency_dfg.png')
pm4py.view_dfg(dfg, start_activities, end_activities)

# Interpretation: the loops on Medicine reconciliation / dispensations and
# Vital sign check reflect repeated clinical work within the stay; their
# weight justifies analysing repetitions per patient group in [NB02-C23].

In [ ]:
# [NB02-C12] PERFORMANCE DFG
# What: discover and visualise the DFG annotated with the MEDIAN elapsed
#       time on each edge (pm4py prints humanised durations).
# Why: the frequency DFG says where the flow goes, the performance DFG
#      says where the flow waits; median is used instead of mean because
#      the durations are right-skewed (NB01).
# Expected: the heaviest median times on the transitions toward Discharge.

# Discover the performance DFG
perf_dfg, start_activities, end_activities = pm4py.discover_performance_dfg(abstract_log)

# Save the figure for the report (median aggregation), then view it inline
pm4py.save_vis_performance_dfg(perf_dfg, start_activities, end_activities,
                               'figures/NB02_performance_dfg.png', aggregation_measure='median')
pm4py.view_performance_dfg(perf_dfg, start_activities, end_activities, aggregation_measure='median')

# Interpretation: reading this graph together with [NB02-C08] and the
# milestone timeline [NB02-C09] localises the bottlenecks; the group
# comparisons below check whether they hit every patient group equally.

In [ ]:
# [NB02-C13] HOW MANY VARIANTS DOES THE PROCESS EXPRESS?
# What: count the distinct sequential variants (exact activity sequences)
#       on both views of the log.
# Why: this is a hospital process, so a very high number of variants is
#      expected; measuring the variability on both views separates real
#      path diversity from the artificial diversity created by repeated
#      same-instant recordings.
# Expected: hundreds of variants; fewer on the abstract view.

# Helper: the sequence of activities of each stay, as a tuple
def case_sequence(values):
    return tuple(values)

# Variants on the raw view
raw_sequences = raw_log.groupby('case:concept:name')['concept:name'].agg(case_sequence)
print('Distinct variants on the raw view: {}'.format(raw_sequences.nunique()))

# Variants on the abstract view
abstract_sequences = abstract_log.groupby('case:concept:name')['concept:name'].agg(case_sequence)
print('Distinct variants on the abstract view: {}'.format(abstract_sequences.nunique()))

# Interpretation: the gap between the two counts is the variability that
# comes only from repeated recordings at the same instant; the abstract
# view is therefore the right basis for variant analysis and discovery.

In [ ]:
# [NB02-C14] SEQUENTIAL VARIANTS TABLE WITH CUMULATIVE COVERAGE
# What: build the table of the abstract-view variants with frequency,
#       share and cumulative share of cases, plus the median stay duration
#       of each variant.
# Why: cumulative coverage tells how many variants are needed to describe
#      the majority of the process - the key question for a readable model
#      in NB03. NOTE: nothing is filtered here; the table describes ALL
#      the variants.
# Expected: with a hospital process, NO strong concentration: many
#      variants needed even for 50% coverage.

# Count the cases of each variant
variants_df = abstract_sequences.value_counts().rename_axis('sequence').reset_index(name='cases')

# Give each variant a readable id, share and cumulative share
variants_df['variant_id'] = ['V{:03d}'.format(i) for i in range(1, len(variants_df) + 1)]
variants_df['share'] = variants_df['cases'] / variants_df['cases'].sum()
variants_df['cumulative_share'] = variants_df['share'].cumsum()

# Median stay duration (minutes) of each variant
sequences_with_duration = abstract_sequences.rename('sequence').to_frame().join(case_table['lead_time_min'])
variant_durations = sequences_with_duration.groupby('sequence')['lead_time_min'].median().rename('median_lead_time_min')
variants_df = variants_df.merge(variant_durations, on='sequence')

# Readable label for the report
def sequence_label(seq):
    return ' -> '.join(seq)
variants_df['sequence_label'] = variants_df['sequence'].apply(sequence_label)

# How many variants cover 50 / 80 / 90 / 100% of the cases?
for target in [0.50, 0.80, 0.90, 1.00]:
    needed = int((variants_df['cumulative_share'] < target).sum()) + 1
    needed = min(needed, len(variants_df))
    print('Variants needed to cover {:.0f}% of the cases: {}'.format(target * 100, needed))

# Save the complete table (ALL variants, none dropped)
variants_df.drop(columns=['sequence']).to_csv('sequential_variants.csv', index=False)
print('\nSaved sequential_variants.csv ({} variants)'.format(len(variants_df)))

variants_df.drop(columns=['sequence']).head(10)

In [ ]:
# [NB02-C15] COVERAGE TABLE (COURSE FORMAT)
# What: the coverage of cases as a function of the number of variants, in
#       the same format used in the course slides (Road Traffic Fines).
# Why: it makes the concentration of the process immediately readable for
#      the report and informs the top-k discussion for NB03 - a decision
#      that will be proposed to the project owner, NOT taken here.
# Expected: slow-growing coverage, confirming the high variability.

# Build the coverage table at increasing numbers of variants
coverage_rows = []
for k in [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 100, len(variants_df)]:
    k = min(k, len(variants_df))
    covered_cases = int(variants_df['cases'].head(k).sum())
    coverage = variants_df['cumulative_share'].iloc[k - 1]
    coverage_rows.append({'number_of_variants': k,
                          'cases_covered': covered_cases,
                          'coverage_of_cases_pct': round(coverage * 100, 1)})
coverage_table = pd.DataFrame(coverage_rows).drop_duplicates('number_of_variants')

# Save the table for the report
coverage_table.to_csv('variant_coverage.csv', index=False)
print('Saved variant_coverage.csv')

coverage_table

In [ ]:
# [NB02-C16] DISTRIBUTION OF THE VARIANTS: POWER-LAW FIT AND SIGNAL OF SEPARATION
# What: fit a power law to the variant frequencies, report the exponent
#       alpha and the Kolmogorov-Smirnov distance D, and compare the power
#       law against the lognormal with the likelihood ratio test.
# Why: course method (Distribution of Variants): sampling/top-k strategies
#      assume a heavy tail; the fit and the separation test tell whether
#      that assumption holds. Interpretation rules from the slides:
##     - alpha: exponent of the fitted power law (typical empirical 2-3)
##     - KS D: distance between data and fitted model (smaller = closer)
##     - R > 0 favours the power law, R < 0 the lognormal
##     - p < 0.1: the signal of separation is significant, choose the
##       model indicated by R; larger p: inconclusive, both fit equally.
# Expected: alpha in the typical range but with a separation signal to
#      check carefully - the coverage table already suggests a weak head.

# Frequencies of the variants, sorted
freqs = variants_df['cases'].sort_values(ascending=False).values

# Log-log plot of rank vs frequency (visual heavy-tail check)
plt.figure(figsize=(8, 5))
plt.loglog(range(1, len(freqs) + 1), freqs, marker='o', linestyle='none', markersize=3)
plt.xlabel('Variant rank (log scale)')
plt.ylabel('Cases per variant (log scale)')
plt.title('Variant frequency distribution (log-log)')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB02_variant_loglog.png', dpi=200)
plt.show()

# Fit the power law on the discrete frequencies (verbose=False suppresses
# the xmin-candidates progress bar, which is easy to misread as a result)
fit = powerlaw.Fit(freqs, discrete=True, verbose=False)
print('Power-law exponent alpha = {:.2f} (xmin = {})'.format(fit.alpha, fit.xmin))
print('Kolmogorov-Smirnov distance D of the power-law fit = {:.3f}'.format(fit.power_law.D))

# Likelihood ratio test: power law vs lognormal (signal of separation)
R, p = fit.distribution_compare('power_law', 'lognormal')
print('Likelihood ratio R = {:.3f}, p = {:.3f}'.format(R, p))
if p < 0.1 and R > 0:
    print('Signal of separation significant (p < 0.1) and R > 0: the power law is the more likely model.')
elif p < 0.1 and R < 0:
    print('Signal of separation significant (p < 0.1) and R < 0: the LOGNORMAL is the more likely model - not a clean Pareto.')
else:
    print('p >= 0.1: the test is inconclusive, both models explain the data equally well.')

# Follow-up test on the sub-head region, lognormal vs POISSON.
# Why here: segment C alone is degenerate for a distribution fit (all its
# frequencies equal 1, there is nothing to fit), so the meaningful question
# is asked on the whole sub-head region (torso + long tail, frequencies
# 1-19): are the rare variants homogeneous random events (Poisson) or
# structural heterogeneity (lognormal)? The powerlaw package has no Poisson
# model, so we compare log-likelihoods with scipy and use AIC (declared
# approximation: the lognormal is continuous, applied to discrete counts).
tail_freqs = freqs[freqs < 20]
print('\nSub-head region (torso + long tail): {} variants, frequencies 1-{}'.format(
    len(tail_freqs), int(tail_freqs.max())))

# Poisson fit: shifted by 1 because frequencies start at 1, not 0
shifted = tail_freqs - 1
mu = shifted.mean()
loglik_poisson = poisson.logpmf(shifted, mu).sum()
aic_poisson = 2 * 1 - 2 * loglik_poisson          # 1 parameter (mu)

# Lognormal fit on the same values (continuous approximation)
shape, loc, scale = lognorm.fit(tail_freqs, floc=0)
loglik_lognorm = lognorm.logpdf(tail_freqs, shape, loc, scale).sum()
aic_lognorm = 2 * 2 - 2 * loglik_lognorm          # 2 parameters (shape, scale)

print('Poisson (shifted): log-likelihood = {:.1f}, AIC = {:.1f}'.format(loglik_poisson, aic_poisson))
print('Lognormal: log-likelihood = {:.1f}, AIC = {:.1f}'.format(loglik_lognorm, aic_lognorm))
if aic_lognorm < aic_poisson:
    print('The lognormal has the lower AIC: the rare variants reflect structural heterogeneity, not homogeneous random noise.')
else:
    print('The Poisson has the lower AIC: the rare variants are compatible with homogeneous random events.')

# Interpretation: alpha ~ 2.1 sits in the typical heavy-tail range, BUT the
# likelihood ratio favours the lognormal at the 10% level (R < 0, p < 0.1):
# the variant distribution has a heavy-ish tail without clean power-law
# behaviour, and the AIC comparison on the sub-head region tells whether
# the long tail behaves like noise (Poisson) or like genuinely diverse
# clinical paths (lognormal). Practical consequence (course, Case8
# 'Distribution Model Invalidation'): an "80/20" top-k sampling is NOT
# supported by the data; any representative-selection choice for NB03 must
# rely on the coverage table [NB02-C15] and be approved by the project
# owner first.

In [ ]:
# [NB02-C17] VARIANT DISTRIBUTION SEGMENTS: HEAD, TORSO AND LONG TAIL
# What: split the variants into the three segments of the course slides -
#       head (dominant paths), torso (recurrent) and long tail (edge
#       cases) - and measure the size and coverage of each segment.
# Why: each segment supports different actions (course, 'Variants:
#      actionable results'): the head is where small improvements have a
#      huge aggregate impact; the long tail is where errors and legitimate
#      complex cases hide - to investigate, never to delete blindly.
# Expected: a thin head and a very large tail of near-unique variants.

# SEGMENT RULE AND ITS JUSTIFICATION (declared analyst choice).
# Head = at least 20 cases | torso = 2-19 cases | long tail = 1 case.
#
# Why 20, and not a number taken from the distribution fit? Because the
# threshold must serve the analysis that uses it, not describe the curve:
#
# 1. Statistical reason: the head variants are the ones we read INSIDE
#    ([NB02-C18] computes median and p90 within each variant). With 20
#    observations the p90 is the second-highest value and is still
#    interpretable; with 3-4 observations it would be noise dressed up as
#    a percentile.
# 2. Business reason: 20 stays is ~1.1% of the ED volume of this log. A
#    path travelled by at least that many patients is one where a process
#    change produces a measurable aggregate effect on the KPIs of
#    [NB02-C05] - the course reading of the head segment ("a small speed
#    gain here impacts a huge number of sessions"). Below that, an
#    intervention would touch too few patients to be monitored.
# 3. Clinical governance reason: a route followed by 20+ patients is a
#    recognisable routine that ED clinicians can discuss as standard
#    practice; a route with 2-3 patients is an anecdote, to be examined
#    case by case (which is exactly the long-tail treatment).
#
# The lower boundary instead is NOT arbitrary: single-case variants sit
# below xmin = 2 of the fit in [NB02-C16], i.e. outside the region where
# the fitted distribution holds - the statistical noise floor.
#
# PENDING CROSS-CHECK (NB03): this threshold is provisional and will be
# challenged with two independent tests - find_best_k_by_f1 (does a
# task-driven k, chosen by model F1, land near the size of this head?)
# and a hypothesis test on whether the three segments really differ in
# lead time (Kruskal-Wallis). Convergence would confirm the choice; a
# divergence would be reported and the threshold revised.
def variant_segment(cases):
    if cases >= 20:
        return 'A - head'
    if cases >= 2:
        return 'B - torso'
    return 'C - long tail'
variants_df['segment'] = variants_df['cases'].apply(variant_segment)

# Size and coverage of each segment
segment_summary = variants_df.groupby('segment').agg(
    variants=('variant_id', 'size'),
    cases=('cases', 'sum'))
segment_summary['coverage_pct'] = (segment_summary['cases'] / segment_summary['cases'].sum() * 100).round(1)
print(segment_summary)

# Business reading, adapted from the course slides to the ED:
## A - head: the standard clinical pathways. Actions: optimise speed and
##     reliability here (small gains x many patients), align staffing,
##     protocols and patient communication with these paths.
## B - torso: recurrent but not dominant situations. Actions: candidates
##     for standardisation - can some torso paths be brought back to a
##     head path without clinical loss?
## C - long tail: edge cases occurring once. Actions: investigate, never
##     delete (no-data-loss rule): each is either a recording anomaly, an
##     interrupted pathway or a genuinely complex patient. The strategic
##     question (course): eliminate the complexity where it is an error,
##     support it where it is legitimate care. NB03 (deviations) and NB04
##     (context patterns) answer it case by case.

# Export the SELECTION LISTS of segments A and C for NB03 (requested by the
# project owner): stay id + variant id + segment. These are pointers into
# the full log, not filtered copies - no data is lost or duplicated.
# Maps from the activity sequence to variant id and segment
variant_id_map = dict(zip(variants_df['sequence'], variants_df['variant_id']))
segment_map = dict(zip(variants_df['sequence'], variants_df['segment']))

# One row per stay with its variant id and segment
stay_segments = sequences_with_duration.copy()
stay_segments['variant_id'] = stay_segments['sequence'].map(variant_id_map)
stay_segments['segment'] = stay_segments['sequence'].map(segment_map)

# Does the head/torso/tail split also differ in SPREAD, not just in the
# median lead time already tested in [NB03-C11]? Levene's test (robust to
# non-normal data, as decided in NB01) answers it.
from scipy.stats import levene
seg_a_dur = stay_segments[stay_segments['segment'] == 'A - head']['lead_time_min']
seg_b_dur = stay_segments[stay_segments['segment'] == 'B - torso']['lead_time_min']
seg_c_dur = stay_segments[stay_segments['segment'] == 'C - long tail']['lead_time_min']
lev_stat, lev_p = levene(seg_a_dur, seg_b_dur, seg_c_dur)
print('\nSpread of lead time within each segment (Levene F = {:.1f}, p = {:.4f}):'.format(lev_stat, lev_p))
for name, dur in [('A - head', seg_a_dur), ('B - torso', seg_b_dur), ('C - long tail', seg_c_dur)]:
    print('  {}: n={} std={:.0f} min, IQR={:.0f} min'.format(name, len(dur), dur.std(), dur.quantile(0.75) - dur.quantile(0.25)))
print('The segments differ in spread as well as in median: the head is a tight, '
      'predictable routine (std {:.0f} min); the long tail is not just slower on '
      'average, it is far more heterogeneous (std {:.0f} min) - a statistical '
      'confirmation of why it is investigated case by case, not standardised.'.format(
          seg_a_dur.std(), seg_c_dur.std()))

stay_segments = stay_segments.drop(columns=['sequence', 'lead_time_min'])
stay_segments.index.name = 'case:concept:name'

# Segment A: the head variants (input for the readable-model candidate of NB03)
segment_a = stay_segments[stay_segments['segment'] == 'A - head']
segment_a.to_csv('segment_A_cases.csv')
print('Saved segment_A_cases.csv ({} stays)'.format(len(segment_a)))

# Segment C: the single-case variants (input for the deviation and context
# analyses of NB03/NB04)
segment_c = stay_segments[stay_segments['segment'] == 'C - long tail']
segment_c.to_csv('segment_C_cases.csv')
print('Saved segment_C_cases.csv ({} stays)'.format(len(segment_c)))

segment_summary

In [ ]:
# [NB02-C18] SAME PATH, DIFFERENT PERFORMANCE: KPI DISPERSION WITHIN VARIANTS
# What: for the head variants (>= 20 cases), measure how much the stay
#       duration varies INSIDE the same variant, i.e. among stays with the
#       IDENTICAL activity sequence.
# Why: patients following the same clinical path should take similar time;
#      a large dispersion within a variant means the time - not the path -
#      makes the difference. This isolates the incidence of waiting in the
#      care: same therapy, different outcome purely because of timing.
# Expected: head variants with p90/median ratios of 2x or more.

# Stays of the head variants only (comparison group; nothing is filtered
# from the data - this only selects what to LOOK at)
head_sequences = variants_df[variants_df['segment'] == 'A - head']['sequence']
head_cases = sequences_with_duration[sequences_with_duration['sequence'].isin(head_sequences)]

# Duration statistics inside each head variant
dispersion = head_cases.groupby('sequence')['lead_time_min'].agg(['size', 'median', p90]).reset_index()
dispersion.columns = ['sequence', 'cases', 'median_min', 'p90_min']

# Ratio p90/median: how much slower is the slow tail of the SAME path?
dispersion['p90_over_median'] = (dispersion['p90_min'] / dispersion['median_min']).round(2)

# Attach the variant id and sort by the dispersion ratio
dispersion = dispersion.merge(variants_df[['sequence', 'variant_id']], on='sequence')
dispersion = dispersion.drop(columns=['sequence']).sort_values('p90_over_median', ascending=False).reset_index(drop=True)

# Save the table for the report
dispersion.to_csv('variant_kpi_dispersion.csv', index=False)
print('Saved variant_kpi_dispersion.csv')

dispersion

# Interpretation: ratios well above 2 mean that within the SAME sequence of
# activities one patient in ten waits more than twice the typical time: the
# path is identical, the clock is not. These variants are the primary
# candidates for the improvement backlog (NB05), because fixing them does
# not require changing the clinical pathway, only its timing.

In [ ]:
# [NB02-C19] SAME PATH, DIFFERENT PATIENTS: PROFILE MIX WITHIN VARIANTS
# What: for the head variants, look at WHO follows them: the acuity and
#       disposition composition of each variant.
# Why: mirror question of [NB02-C18]: when very different patients share
#      the same activity sequence, the process treats them alike - which
#      may be fine (a standard intake path) or a red flag (urgency not
#      reflected in the treatment path). The age attribute used in the
#      assignment example is not present in this dataset (declared
#      limitation), so acuity and disposition are the patient profile.
# Expected: head variants shared by several acuity levels.

# Map each stay of the head variants to its variant id
variant_id_map = dict(zip(variants_df['sequence'], variants_df['variant_id']))
head_profile = sequences_with_duration[sequences_with_duration['sequence'].isin(head_sequences)].copy()
head_profile['variant_id'] = head_profile['sequence'].map(variant_id_map)

# Attach the patient profile attributes from the case table
head_profile = head_profile.join(case_table[['acuity', 'disposition']])

# Acuity composition of each head variant (rows sum to 1)
acuity_mix = pd.crosstab(head_profile['variant_id'], head_profile['acuity'], normalize='index').round(2)
print('Acuity composition of the head variants:')
print(acuity_mix)

# Disposition composition of the same variants (rows sum to 1)
disposition_mix = pd.crosstab(head_profile['variant_id'], head_profile['disposition'], normalize='index').round(2)
print('\nDisposition composition of the head variants:')
print(disposition_mix)

# Interpretation: variants whose acuity row spreads across many levels are
# paths where patients with different urgency receive the same sequence of
# steps - the same clinical route for a critical and a minor patient. The
# report raises this as a governance discussion point, not a conclusion;
# NB04 will make it measurable with the binary context patterns.

In [ ]:
# [NB02-C20] STAY DURATION BY ACUITY
# What: compare the stay duration across the five acuity levels with a
#       Kruskal-Wallis test and a boxplot.
# Why: acuity is the triage urgency (Appendix: 1-3 = urgent); if urgent
#      patients also wait longer, the flow contradicts its own priorities
#      (KPI 'Urgent-path timeliness', [NB02-C05]). The 56 stays without a
#      recorded acuity are NOT dropped: they simply cannot appear in a
#      by-acuity comparison (denominator declared).
# Expected: different duration profiles across acuity levels.

# Stays with a recorded acuity (the others are excluded from THIS
# comparison only - they remain in every other analysis)
with_acuity = case_table[case_table['acuity'].notna()]
print('Stays with recorded acuity: {} of {} (56 without remain in all other analyses)'.format(
    len(with_acuity), len(case_table)))

# Median, p90 AND std duration (minutes) per acuity level, with group sizes
by_acuity = with_acuity.groupby('acuity')['lead_time_min'].agg(['size', 'median', p90, 'std'])
by_acuity.columns = ['stays', 'median_min', 'p90_min', 'std_min']
print('\nStay duration by acuity (minutes):')
print(by_acuity)

# Kruskal-Wallis test across the acuity groups (non-parametric, as decided in NB01)
groups = []
for level, group in with_acuity.groupby('acuity'):
    groups.append(group['lead_time_min'].dropna().values)
stat, p_value = kruskal(*groups)
print('\nKruskal-Wallis H = {:.2f}, p-value = {:.4f}'.format(stat, p_value))
if p_value < 0.05:
    print('Stay duration differs significantly across acuity levels.')
else:
    print('No significant duration difference across acuity levels.')

# Does the spread differ too, not just the median? Levene's test (robust to
# non-normal data, as decided in NB01) - the ANOVA-style variance-equality
# check that does not assume normality.
from scipy.stats import levene
lev_stat, lev_p = levene(*groups)
print('\nLevene F = {:.2f}, p = {:.4f}: acuity 2 has std={:.0f} min, the widest of'
      ' all five levels - it is not only the slowest median, it is also the'
      ' least predictable.'.format(lev_stat, lev_p, by_acuity.loc[2.0, 'std_min']))

# Boxplot of the duration by acuity (hours for readability; outliers hidden
# from the PLOT only - they stay in the data and in the statistics)
plt.figure(figsize=(9, 5))
sns.boxplot(data=with_acuity, x='acuity', y=with_acuity['lead_time_min'] / 60, showfliers=False, color='#9dc3e6')
plt.xlabel('Acuity (ESI, 1 = most urgent)')
plt.ylabel('Stay duration (hours)')
plt.title('Stay duration by acuity level')
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB02_duration_by_acuity.png', dpi=200)
plt.show()

# Interpretation: acuity 2 shows the LONGEST median stay - longer than the
# most critical level 1: the very urgent are fast-tracked, but the
# "serious yet stable" patients absorb the longest stays. This nuance
# matters for the urgent-path timeliness KPI and returns in NB04.

In [ ]:
# [NB02-C21] STAY DURATION BY ARRIVAL MODE, DISPOSITION AND INTERRUPTED PATHWAYS
# What: pairwise comparisons of the stay duration for the main business
#       segments: ambulance vs walk-in, admitted vs home, interrupted vs
#       regular pathways. Beyond the median gap, also check whether the two
#       groups are equally SPREAD OUT (variance), not just shifted.
# Why: these are the groups behind the business rules of the report
#      (ambulance = pre-hospital urgency; admitted = clinically heavier);
#      Mann-Whitney U is used because durations are not normal (NB01).
#      Levene's test is the variance-equality analogue of an ANOVA F-test
#      that does not assume normality - it answers "is one group also more
#      HETEROGENEOUS than the other?", a question the median/Wasserstein
#      comparison alone cannot answer.
# Expected: admitted stays clearly longer than home discharges.

# Helper: compare two groups with a significance test, an effect-size
# distance AND a spread comparison. Mann-Whitney U answers "do they
# differ?"; the Wasserstein distance (course note) answers "by how much?",
# measuring the distance between the two full cumulative distributions in
# minutes; IQR/std quantify each group's own spread; Levene's test (the
# non-parametric-safe F-test for variance equality) answers "is one group
# also more heterogeneous?".
from scipy.stats import wasserstein_distance, levene
def compare_groups(name_a, values_a, name_b, values_b):
    stat, p_value = mannwhitneyu(values_a, values_b, alternative='two-sided')
    w_distance = wasserstein_distance(values_a, values_b)
    iqr_a = values_a.quantile(0.75) - values_a.quantile(0.25)
    iqr_b = values_b.quantile(0.75) - values_b.quantile(0.25)
    lev_stat, lev_p = levene(values_a, values_b)
    print('{} (n={}, median {:.0f} min, IQR {:.0f}, std {:.0f}) vs {} (n={}, median {:.0f} min, IQR {:.0f}, std {:.0f}):'.format(
        name_a, len(values_a), np.median(values_a), iqr_a, values_a.std(),
        name_b, len(values_b), np.median(values_b), iqr_b, values_b.std()))
    print('  Mann-Whitney U = {:.0f}, p = {:.4f} | Wasserstein = {:.0f} min | Levene F (equal variance) p = {:.4f}{}'.format(
        stat, p_value, w_distance, lev_p, ' -> variances differ too' if lev_p < 0.05 else ' -> variances not significantly different'))

# Ambulance vs walk-in arrivals
ambulance = case_table[case_table['arrival_transport'] == 'AMBULANCE']['lead_time_min'].dropna()
walkin = case_table[case_table['arrival_transport'] == 'WALK IN']['lead_time_min'].dropna()
compare_groups('AMBULANCE', ambulance, 'WALK IN', walkin)

# Admitted vs discharged home
admitted = case_table[case_table['disposition'] == 'ADMITTED']['lead_time_min'].dropna()
home = case_table[case_table['disposition'] == 'HOME']['lead_time_min'].dropna()
compare_groups('ADMITTED', admitted, 'HOME', home)

# Interrupted pathways vs regular pathways (flag built in NB01-C13)
interrupted = case_table[case_table['interrupted_pathway']]['lead_time_min'].dropna()
regular = case_table[~case_table['interrupted_pathway']]['lead_time_min'].dropna()
compare_groups('INTERRUPTED', interrupted, 'REGULAR', regular)

# Interpretation: each line reports group sizes (denominators), medians in
# minutes, the location test (Mann-Whitney) and the spread test (Levene).
# Ambulance arrivals are not only ~80 min longer in median than walk-in,
# they are ALSO significantly more heterogeneous (Levene p < 0.001, std 494
# vs 374 min) - a wider mix of clinical severity among ambulance patients.
# Admitted-vs-home and interrupted-vs-regular differ in median but NOT in
# spread: those two segments are shifted, not more variable. The effect
# direction (which group is LONGER, not just how far apart) matters as much
# as the p-value and is discussed in the report.

In [ ]:
# [NB02-C22] ARRIVALS AND DURATION BY HOUR OF DAY
# What: distribution of the arrivals over the 24 hours and median stay
#       duration by hour of arrival.
# Why: ED load is strongly time-dependent; if the duration grows in
#      specific arrival windows, staffing is a candidate lever (NB05).
#      Calendar dates are de-identified (year 2110) but hours of day and
#      durations remain meaningful.
# Expected: morning/afternoon arrival peaks.

# Hour of arrival of each stay (0-23)
case_table['arrival_hour'] = case_table['start_time'].dt.hour

# Arrivals per hour of day
arrivals_per_hour = case_table['arrival_hour'].value_counts().sort_index()

# Median stay duration (minutes) by hour of arrival
median_by_hour = case_table.groupby('arrival_hour')['lead_time_min'].median()

# Two-panel figure: arrivals and median duration by hour
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(arrivals_per_hour.index, arrivals_per_hour.values, color='#5b9bd5', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Hour of arrival (0-23)')
axes[0].set_ylabel('Number of stays')
axes[0].set_title('Arrivals by hour of day')
axes[1].plot(median_by_hour.index, median_by_hour.values, marker='o', color='#c55a11')
axes[1].set_xlabel('Hour of arrival (0-23)')
axes[1].set_ylabel('Median stay duration (min)')
axes[1].set_title('Median stay duration by hour of arrival')
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB02_hour_of_day.png', dpi=200)
plt.show()

# Interpretation: the two panels are read together: hours with many
# arrivals AND high median duration are the first candidates for a
# staffing/fast-track discussion in the improvement backlog (NB05).

In [ ]:
# [NB02-C23] REPETITIONS OF CLINICAL ACTIVITIES PER STAY
# What: count medicine reconciliations, dispensations and vital sign
#       checks per stay (raw view - every recording counts) and compare
#       the counts across dispositions.
# Why: this is the clinical workload KPI (cost proxy, [NB02-C05]): if
#      repetitions concentrate on specific outcomes, they explain the
#      duration gaps seen above.
# Expected: admitted stays with more clinical recordings than home stays.

# Count each clinical activity per stay on the RAW view (all recordings)
clinical_activities = ['Medicine reconciliation', 'Medicine dispensations', 'Vital sign check']
for activity in clinical_activities:
    activity_rows = raw_log[raw_log['concept:name'] == activity]
    counts = activity_rows.groupby('case:concept:name').size()
    column_name = activity.lower().replace(' ', '_') + '_count'
    case_table[column_name] = counts
    case_table[column_name] = case_table[column_name].fillna(0).astype(int)

# Median counts by disposition for the two main outcomes
main_dispositions = case_table[case_table['disposition'].isin(['HOME', 'ADMITTED'])]
count_columns = ['medicine_reconciliation_count', 'medicine_dispensations_count', 'vital_sign_check_count']
print('Median clinical recordings per stay, by disposition:')
print(main_dispositions.groupby('disposition')[count_columns].median())

# And by urgency (Appendix flag acuity 1-3 from NB01)
print('\nMedian clinical recordings per stay, urgent (acuity 1-3) vs not:')
print(case_table.groupby('acuity_1_3')[count_columns].median())

# Interpretation: the medians quantify how much clinical work each segment
# absorbs; large gaps here anticipate the deviation analysis of NB03 and
# the context patterns of NB04 (repeated_vitals, repeated_medication).

In [ ]:
# [NB02-C24] EQUITY MONITORING BY GENDER AND RACE (GOAL S2)
# What: test whether stay duration, admission rate and the interrupted-
#       pathway rate differ by gender and by race, and check whether the
#       acuity mix (case severity) differs too - a possible confound.
# Why: goal S2 promises equity monitoring and the case-study description
#      declares gender/race as fields "used only for equity monitoring...
#      never for clinical inference" - but until this cell that promise was
#      never actually tested. Testing it, not assuming it, is the same
#      project rule applied everywhere else. Kruskal-Wallis/Mann-Whitney
#      for the skewed duration (as decided in NB01), chi-square for the
#      categorical rates.
# Expected: unknown going in - this is an honest check, not a foregone
#      conclusion in either direction.

from scipy.stats import chi2_contingency
interrupted_labels = ['LEFT WITHOUT BEING SEEN', 'ELOPED', 'LEFT AGAINST MEDICAL ADVICE', 'EXPIRED', 'OTHER']

# ---- Gender: two groups, so Mann-Whitney + chi-square are enough
print('=' * 60)
print('GENDER')
print('=' * 60)
male = case_table[case_table['gender'] == 'M']['lead_time_min'].dropna()
female = case_table[case_table['gender'] == 'F']['lead_time_min'].dropna()
stat_g, p_g = mannwhitneyu(male, female, alternative='two-sided')
print('Duration, M (n={}, median {:.0f} min) vs F (n={}, median {:.0f} min): p = {:.3f}'.format(
    len(male), male.median(), len(female), female.median(), p_g))

tab_admit_gender = pd.crosstab(case_table['gender'], case_table['disposition'] == 'ADMITTED')
chi2_ag, p_admit_gender, _, _ = chi2_contingency(tab_admit_gender)
print('Admission rate by gender: M {:.1f}% vs F {:.1f}%, chi2 p = {:.3f}'.format(
    (case_table[case_table['gender'] == 'M']['disposition'] == 'ADMITTED').mean() * 100,
    (case_table[case_table['gender'] == 'F']['disposition'] == 'ADMITTED').mean() * 100, p_admit_gender))

tab_inter_gender = pd.crosstab(case_table['gender'], case_table['disposition'].isin(interrupted_labels))
chi2_ig, p_inter_gender, _, _ = chi2_contingency(tab_inter_gender)
print('Interrupted-pathway rate by gender: p = {:.3f} (the S2/T2 quality KPI of Table 4)'.format(p_inter_gender))

tab_acuity_gender = pd.crosstab(case_table['gender'], case_table['acuity'].isin([1, 2, 3]))
chi2_acg, p_acuity_gender, _, _ = chi2_contingency(tab_acuity_gender)
print('Confound check - acuity-1-3 share by gender: chi2 p = {:.3f}'.format(p_acuity_gender))

# ---- Race: many small categories; keep only groups with n >= 20 (the same
# reasoning as the n >= 20 variant-head threshold of NB02-C17), and treat
# UNKNOWN as a data-completeness flag, not a demographic group
print()
print('=' * 60)
print('RACE (groups with n >= 20; UNKNOWN excluded from the statistical test)')
print('=' * 60)
race_counts = case_table['race'].value_counts()
race_groups_ok = race_counts[race_counts >= 20].index.tolist()
race_for_test = [r for r in race_groups_ok if r != 'UNKNOWN']
print('Race groups tested (n >= 20, excluding UNKNOWN):', race_for_test)

by_race = case_table[case_table['race'].isin(race_for_test)]
race_duration_groups = [by_race[by_race['race'] == r]['lead_time_min'].dropna() for r in race_for_test]
stat_r, p_r = kruskal(*race_duration_groups)
print('\nDuration by race, Kruskal-Wallis: H = {:.1f}, p = {:.4f}'.format(stat_r, p_r))
print(by_race.groupby('race')['lead_time_min'].agg(['size', 'median']).loc[race_for_test])

tab_admit_race = pd.crosstab(by_race['race'], by_race['disposition'] == 'ADMITTED')
chi2_ar, p_admit_race, _, _ = chi2_contingency(tab_admit_race)
print('\nAdmission rate by race: chi2 p = {:.3f}'.format(p_admit_race))

tab_inter_race = pd.crosstab(by_race['race'], by_race['disposition'].isin(interrupted_labels))
chi2_ir, p_inter_race, _, _ = chi2_contingency(tab_inter_race)
print('Interrupted-pathway rate by race: chi2 p = {:.3f} (small subgroups: read with caution)'.format(p_inter_race))

tab_acuity_race = pd.crosstab(by_race['race'], by_race['acuity'].isin([1, 2, 3]))
chi2_acr, p_acuity_race, _, _ = chi2_contingency(tab_acuity_race)
print('Confound check - acuity-1-3 share by race: chi2 p = {:.3f}'.format(p_acuity_race))

# Export for the report table
equity_summary = by_race.groupby('race').agg(
    stays=('lead_time_min', 'size'),
    median_lead_min=('lead_time_min', 'median'),
    admitted_pct=('disposition', lambda s: (s == 'ADMITTED').mean() * 100),
    acuity_1_3_pct=('acuity', lambda s: s.isin([1, 2, 3]).mean() * 100)).loc[race_for_test].round(1)
equity_summary.to_csv('equity_monitoring_by_race.csv')
equity_summary

# Interpretation: gender is broadly balanced - duration does NOT differ
# (p = 0.685) and the declared quality KPI (interrupted-pathway rate) does
# NOT differ either (p = 0.767); the small admission-rate gap coincides
# with a small but significant difference in acuity mix (p = 0.002), so it
# is monitored, not diagnosed as a gender effect. Race shows a REAL signal:
# duration (p = 0.0021) and admission rate (p = 0.013) differ significantly
# across the nine groups with n >= 20 (UNKNOWN excluded as a data-
# completeness flag, not a demographic group) - and, unlike gender, the
# acuity-mix confound check is NOT significant here (p = 0.148): the gap is
# NOT explained by these groups being triaged at different urgency levels.
# The interrupted-pathway rate does not differ significantly (p = 0.350,
# small subgroups). This is exactly what the S2 goal asks to monitor; the
# log carries no clinical or social information that could explain the
# duration/admission gap, so it is reported as a quantified, non-causal
# equity signal for operational follow-up, never as a diagnosis of bias.

In [ ]:
# [NB02-C25] EXPORTS AND NO-DATA-LOSS CHECK
# What: recap the artifacts produced by this notebook and verify that no
#       data was lost anywhere in the analysis.
# Why: project rule - nothing is ever dropped; every notebook ends by
#      proving it.
# Expected: the same row and stay counts we started with.

# Verify: the three views still have all their rows and stays
print('Raw view rows: {} (started with 25,115)'.format(len(raw_log)))
print('Abstract view rows: {} (started with 16,826)'.format(len(abstract_log)))
print('Stays in the case table: {} (started with 1,820)'.format(len(case_table)))

# Save the case table enriched with the counts of this notebook
case_table.to_csv('case_table.csv')
print('\nUpdated case_table.csv with arrival_hour and clinical activity counts')

# Recap of the artifacts
print('\nNB02 outputs: kpi_definitions.csv, transition_performance.csv,')
print('incremental_lead_time.csv, sequential_variants.csv, variant_coverage.csv,')
print('variant_kpi_dispersion.csv, segment_A_cases.csv, segment_C_cases.csv,')
print('equity_monitoring_by_race.csv, updated case_table.csv, 7 figures in figures/')
print('\nNo filter was applied in this notebook. The only pending decision')
print('(representative selection for NB03 discovery) will be proposed to the')
print('project owner first, based on the coverage table and the separation test.')